## Création d'un seul DATAFRAME (fusion IMDB & TMDB)

In [ ]:
import pandas as pd
import numpy as np
import re

# Listes des URL pour récupérer les bases de données

url_titles = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_ratings = 'https://datasets.imdbws.com/title.ratings.tsv.gz'
url_akas = 'https://datasets.imdbws.com/title.akas.tsv.gz'
url_principals = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_crew = 'https://datasets.imdbws.com/title.crew.tsv.gz'
url_names =  'https://datasets.imdbws.com/name.basics.tsv.gz'


# pd.set_option('display.max_colwidth', None)     # Affichage de l'ensemble des caractères dans les colonnes des dataframes


In [ ]:
# Création DF_PRINCIPALS

colonnes = {'tconst': 'Titre_ID', 'nconst': 'Nom_ID' , 'category': 'Categorie'}                       # Sélection des colonnes

df_principals = pd.read_csv(url_principals, sep='\t', usecols=colonnes.keys())                        # création de dataframe

df_principals = df_principals.rename(columns=colonnes)                                                # Colonnes renommées

df_principals.head()


,Titre_ID,Nom_ID,Categorie
0,tt0000001,nm1588970,self
1,tt0000001,nm0005690,director
2,tt0000001,nm0005690,producer
3,tt0000001,nm0374658,cinematographer
4,tt0000002,nm0721526,director


In [ ]:
# Création DF_TITLES

colonnes = {'tconst': 'Titre_ID', 'titleType' : 'Type','primaryTitle': 'Titre', 'isAdult' : 'Adulte', 'startYear' : 'Annee', 'runtimeMinutes' : 'Duree', 'genres' : 'Genre'}

df_titles = pd.read_csv(url_titles, sep='\t', usecols=colonnes.keys()).query("isAdult == 0")

df_titles = df_titles.rename(columns=colonnes)

df_titles.head()

,Titre_ID,Type,Titre,Adulte,Annee,Duree,Genre
0,tt0000001,short,Carmencita,0,1894,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,0,1892,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,0,1892,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,0,1892,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,0,1893,1,Short


In [ ]:
df_titles['Type'].str.contains('Art').value_counts()


Type
False    11936565
Name: count, dtype: int64

In [ ]:
# Création DF_CREW

colonnes = {'tconst': 'Titre_ID', 'directors': 'Realisateur_ID' , 'writers': 'Scenariste_ID'}

df_crew = pd.read_csv(url_crew, sep='\t')

df_crew = df_crew.rename(columns=colonnes)

df_crew.head()

,Titre_ID,Realisateur_ID,Scenariste_ID
0,tt0000001,nm0005690,\N
1,tt0000002,nm0721526,\N
2,tt0000003,nm0721526,nm0721526
3,tt0000004,nm0721526,\N
4,tt0000005,nm0005690,\N


In [ ]:
# Création DF_NAMES

colonnes = {'nconst': 'Nom_ID','primaryName' : 'Nom' ,'primaryProfession' : 'Profession', 'knownForTitles' : 'Titres_ID'}

df_names = pd.read_csv(url_names, sep='\t', usecols=colonnes.keys())

df_names = df_names.rename(columns=colonnes)

df_names.head()

,Nom_ID,Nom,Profession,Titres_ID
0,nm0000001,Fred Astaire,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0117057"
2,nm0000003,Brigitte Bardot,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,"writer,director,actor","tt0050986,tt0069467,tt0050976,tt0083922"


In [ ]:
df_actors = df_principals.merge(df_names, on='Nom_ID',  how='left')                                     # Création d'un dataframe actors

df_actors = df_actors[(df_actors['Categorie'].str.contains('actor|actress'))]                           # sélection des acteurs et actrices

df_actors = df_actors.drop(columns=['Profession', 'Titres_ID'])

df_actors = df_actors.dropna(subset=['Nom'])

df_actors = df_actors.groupby('Titre_ID')['Nom'].agg(', '.join)

df_actors.head()

Titre_ID
tt0000005                             Charles Kayser, John Ott
tt0000007                     James J. Corbett, Peter Courtney
tt0000008                                             Fred Ott
tt0000009    Blanche Bayliss, William Courtenay, Chauncey D...
tt0000011                                              Grunato
Name: Nom, dtype: str

In [ ]:
# Création DF_RATING

colonnes = {'tconst': 'Titre_ID','averageRating' : 'Note', 'numVotes' : 'Nb_votes'}

df_ratings = pd.read_csv(url_ratings, sep='\t')

df_ratings = df_ratings.rename(columns=colonnes)

df_ratings.head()


,Titre_ID,Note,Nb_votes
0,tt0000001,5.7,2197
1,tt0000002,5.5,310
2,tt0000003,6.5,2302
3,tt0000004,5.1,196
4,tt0000005,6.2,3034


In [ ]:
# Création DF_AKAS

colonnes = {'titleId' : 'Titre_ID','title' : 'Titre_FR', 'region' : 'Region', 'language' : 'Langue_Titre'}

df_akas = pd.read_csv(url_akas, sep='\t', usecols=colonnes.keys())

df_akas = df_akas[df_akas['region'] == 'FR']

df_akas = df_akas.drop_duplicates(subset='titleId')                         # Suppression des titres de films en double

df_akas = df_akas.rename(columns=colonnes)

df_akas.head()




,Titre_ID,Titre_FR,Region,Langue_Titre
12,tt0000002,Le clown et ses chiens,FR,\N
23,tt0000003,Pauvre Pierrot,FR,\N
31,tt0000004,Un bon bock,FR,\N
86,tt0000010,La sortie de l'usine,FR,\N
123,tt0000012,L'arrivée d'un train en gare de La Ciotat,FR,\N


In [ ]:
colonnes = {'imdb_id' : 'Titre_ID','spoken_languages': 'Langue', 'original_title' : 'Titre Original', 'revenue': 'Recette', 'vote_average': 'Note_TMDB', 'vote_count' : 'Nb_votes_TMDB', 'adult' : 'Adulte',
             'popularity': 'Popularite','poster_path' : 'Affiche', 'backdrop_path' : 'Image', 'video' : 'Bande_annonce', 'production_countries' : 'Pays', 'overview': 'Resume', 'budget' : 'Budget'}


# !!! ATTENTION !!! Copier la base de donnée TMDB dans le même répertoire que que le fichier si utilisation de VS Code ou l'importer dans le Colab

df_tmdb = pd.read_csv('tmdb_full.csv', sep=',', usecols=colonnes.keys()).query("adult == False")

df_tmdb = df_tmdb.rename(columns=colonnes)

df_tmdb = df_tmdb.drop(columns=['Adulte'])

df_tmdb.head()


,Image,Budget,Titre_ID,Titre Original,Resume,Popularite,Affiche,Pays,Recette,Langue,Bande_annonce,Note_TMDB,Nb_votes_TMDB
0,/dvQj1GBZAZirz1skEEZyWH2ZqQP.jpg,0,tt0029927,Blondie,Blondie and Dagwood are about to celebrate the...,2.852,/zBiHKhXklvTFwj4M1uEUcQGAVJ.jpg,['US'],0,['en'],False,7.214,7
1,NaN,0,tt0011436,Der Mann ohne Namen,NaN,1.091,/6xUbUCvndklbGVYiljHr34NTxSl.jpg,['DE'],0,[],False,0.000,0
2,/uJlc4aNPF3Y8yAqahJTKBwgwPVW.jpg,0,tt0055747,L'Amour à vingt ans,Love at Twenty unites five directors from five...,3.770,/aup2QCYCsyEeQfpboXy0f4uj8aE.jpg,"['DE', 'FR', 'IT', 'JP', 'PL']",0,"['it', 'ja', 'pl', 'fr', 'de']",False,6.700,41
3,/hQ4pYsIbP22TMXOUdSfC2mjWrO0.jpg,0,tt0094675,Ariel,Taisto Kasurinen is a Finnish coal miner whose...,9.214,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,['FI'],0,['fi'],False,7.046,248
4,/l94l89eMmFKh7na2a1u5q67VgNx.jpg,0,tt0092149,Varjoja paratiisissa,"An episode in the life of Nikander, a garbage ...",6.282,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,['FI'],0,['en'],False,7.182,269


In [ ]:
# FUSION DE TOUS LES DATAFRAMES IMDB

df = df_titles.merge(df_ratings, on='Titre_ID', how='left').merge(df_crew, on='Titre_ID', how='left').merge(df_akas, on='Titre_ID', how='left')

df = df.merge(df_names[['Nom_ID', 'Nom']], left_on='Realisateur_ID', right_on='Nom_ID', how='left')

df = df.rename(columns={'Nom': 'Realisateur'}).drop(columns='Nom_ID')

df = df.drop(columns=['Realisateur_ID', 'Scenariste_ID', 'Adulte'])

df = df.merge(df_actors, on='Titre_ID',  how='left')

df = df.rename(columns={'Nom': 'Acteurs'})

df = df.drop_duplicates(subset='Titre_ID')                         # Suppression des titres de films en double

df.head()

,Titre_ID,Type,Titre,Annee,Duree,Genre,Note,Nb_votes,Titre_FR,Region,Langue_Titre,Realisateur,Acteurs
0,tt0000001,short,Carmencita,1894,1,"Documentary,Short",5.7,2197.0,NaN,NaN,NaN,William K.L. Dickson,NaN
1,tt0000002,short,Le clown et ses chiens,1892,5,"Animation,Short",5.5,310.0,Le clown et ses chiens,FR,\N,Émile Reynaud,NaN
2,tt0000003,short,Poor Pierrot,1892,5,"Animation,Comedy,Romance",6.5,2302.0,Pauvre Pierrot,FR,\N,Émile Reynaud,NaN
3,tt0000004,short,Un bon bock,1892,12,"Animation,Short",5.1,196.0,Un bon bock,FR,\N,Émile Reynaud,NaN
4,tt0000005,short,Blacksmith Scene,1893,1,Short,6.2,3034.0,NaN,NaN,NaN,William K.L. Dickson,"Charles Kayser, John Ott"


In [ ]:
# FUSION DES DEUX DATAFRAME (IMDB GLOBAL & TMDB)

df = pd.merge(df, df_tmdb, how="left", on="Titre_ID")

df = df.drop(columns=['Bande_annonce','Note_TMDB','Nb_votes_TMDB'])


In [ ]:
#  df.to_csv("df_KPI.csv", index=False)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11936565 entries, 0 to 11936564
Data columns (total 22 columns):
 #   Column          Dtype  
---  ------          -----  
 0   Titre_ID        str    
 1   Type            str    
 2   Titre           str    
 3   Annee           str    
 4   Duree           str    
 5   Genre           str    
 6   Note            float64
 7   Nb_votes        float64
 8   Titre_FR        str    
 9   Region          str    
 10  Langue_Titre    str    
 11  Realisateur     str    
 12  Acteurs         str    
 13  Image           str    
 14  Budget          float64
 15  Titre Original  str    
 16  Resume          str    
 17  Popularite      float64
 18  Affiche         str    
 19  Pays            str    
 20  Recette         float64
 21  Langue          str    
dtypes: float64(5), str(17)
memory usage: 3.4 GB


In [ ]:
colonnes = df.columns

print(f"Pourcentage de valeurs manquantes par colonne :")

for colonne in colonnes :

    pourcentage = round(((df[colonne].isna().sum()) / len(df)) * 100, 1)

    if pourcentage != 0 :
        print(f"{colonne} : {pourcentage} %")

Pourcentage de valeurs manquantes par colonne :
Note : 86.4 %
Nb_votes : 86.4 %
Titre_FR : 55.1 %
Region : 55.1 %
Langue_Titre : 55.1 %
Realisateur : 54.6 %
Acteurs : 47.3 %
Image : 98.7 %
Budget : 97.4 %
Titre Original : 97.4 %
Resume : 97.6 %
Popularite : 97.4 %
Affiche : 97.8 %
Pays : 97.4 %
Recette : 97.4 %
Langue : 97.4 %


## Sélection et Nettoyage du DATAFRAME FINAL

In [ ]:
# SELECTIONS DES FILMS

condition = ((df['Annee'] >= "1950") & (df['Type']=='movie') & (df['Genre'].str.contains('Action|Adventure|Documentary|Drama|Fantasy|Animation|Comedy|Family|History', na = False)) & (df['Region'].str.contains('FR')))

df_final = df[condition]

df_final = df_final.drop(columns=['Type', 'Region'])



In [ ]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 70223 entries, 21702 to 11936334
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Titre_ID        70223 non-null  str    
 1   Titre           70223 non-null  str    
 2   Annee           70223 non-null  str    
 3   Duree           70223 non-null  str    
 4   Genre           70223 non-null  str    
 5   Note            59722 non-null  float64
 6   Nb_votes        59722 non-null  float64
 7   Titre_FR        70223 non-null  str    
 8   Langue_Titre    70223 non-null  str    
 9   Realisateur     63168 non-null  str    
 10  Acteurs         58822 non-null  str    
 11  Image           40402 non-null  str    
 12  Budget          50752 non-null  float64
 13  Titre Original  50752 non-null  str    
 14  Resume          49229 non-null  str    
 15  Popularite      50752 non-null  float64
 16  Affiche         49203 non-null  str    
 17  Pays            50752 non-null  str    


In [ ]:
colonnes = df_final.columns

print(f"Pourcentage de valeurs manquantes par colonne :")

for colonne in colonnes :

    pourcentage = round(((df_final[colonne].isna().sum()) / len(df_final)) * 100, 1)

    if pourcentage != 0 :
        print(f"{colonne} : {pourcentage} %")

Pourcentage de valeurs manquantes par colonne :
Note : 15.0 %
Nb_votes : 15.0 %
Realisateur : 10.0 %
Acteurs : 16.2 %
Image : 42.5 %
Budget : 27.7 %
Titre Original : 27.7 %
Resume : 29.9 %
Popularite : 27.7 %
Affiche : 29.9 %
Pays : 27.7 %
Recette : 27.7 %
Langue : 27.7 %


In [ ]:
df_final[df_final["Titre_ID"].str.contains('tt0211915')]

,Titre_ID,Titre,Annee,Duree,Genre,Note,Nb_votes,Titre_FR,Langue_Titre,Realisateur,Acteurs,Image,Budget,Titre Original,Resume,Popularite,Affiche,Pays,Recette,Langue
190038,tt0211915,Amélie,2001,122,"Comedy,Romance",8.3,836238.0,Amélie des Abbesses,\N,Jean-Pierre Jeunet,"Audrey Tautou, Mathieu Kassovitz, Rufus, Lorel...",/9Y9K6LeLrMeofOvX7hZW36Aj3OG.jpg,10000000.0,Le Fabuleux Destin d'Amélie Poulain,"At a tiny Parisian café, the adorable yet pain...",26.104,/oTKduWL2tpIKEmkAqF4mFEAWAsv.jpg,"['FR', 'DE']",173921954.0,"['fr', 'ru']"


In [ ]:
# NETTOYAGE DU DATAFRAME

# Suppression des valeurs manquantes

df_final = df_final.dropna()

# Nettoyage des valeurs incohérentes ou non pertinentes

df_final['Titre'] = df_final['Titre'].str.replace(r'[#.%]', '', regex=True).str.strip()                         # Nettoyage des titres des films
df_final['Titre_FR'] = df_final['Titre_FR'].str.replace(r'[#.%]', '', regex=True).str.strip()                   # Nettoyage des titres des films
df_final['Titre Original'] = df_final['Titre Original'].str.replace(r'[#.%]', '', regex=True).str.strip()       # Nettoyage des titres des films


df_final['Titre Original'] = df_final['Titre Original'].fillna(df_final['Titre_FR'])                            # Remplacement des valeurs manquantes de titre Original par un titre Français
df_final['Titre'] = df_final['Titre_FR']                                                                        #
df_final.loc[df_final['Langue'].str.contains('fr'), 'Titre']= df_final['Titre Original']

df_final['Annee'] = df_final['Annee'].replace("\\N","")                                                         # Nettoyer et convertir la colonne Annee
df_final['Annee'] = pd.to_datetime(df_final['Annee'])
df_final['Annee'] = df_final['Annee'].dt.year
df_final['Annee'] = df_final['Annee'].astype(int)

df_final = df_final.sort_values(by='Annee', ascending = False)
df_final = df_final.drop_duplicates(subset='Titre', keep='first')

df_final = df_final.drop(columns=['Titre Original', 'Titre_FR', 'Langue'])


df_final = df_final[df_final['Titre_ID'].str.match(r'^tt\d+$')]                                                 # Nettoyage des lignes non correctes


df_final.info()

<class 'pandas.DataFrame'>
Index: 34222 entries, 11794812 to 41076
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Titre_ID      34222 non-null  str    
 1   Titre         34222 non-null  str    
 2   Annee         34222 non-null  int64  
 3   Duree         34222 non-null  str    
 4   Genre         34222 non-null  str    
 5   Note          34222 non-null  float64
 6   Nb_votes      34222 non-null  float64
 7   Langue_Titre  34222 non-null  str    
 8   Realisateur   34222 non-null  str    
 9   Acteurs       34222 non-null  str    
 10  Image         34222 non-null  str    
 11  Budget        34222 non-null  float64
 12  Resume        34222 non-null  str    
 13  Popularite    34222 non-null  float64
 14  Affiche       34222 non-null  str    
 15  Pays          34222 non-null  str    
 16  Recette       34222 non-null  float64
dtypes: float64(5), int64(1), str(11)
memory usage: 23.7 MB


In [ ]:
df_final.to_csv("df_final.csv", index=False)

In [ ]:
df_final[df_final["Titre_ID"].str.contains('tt0211915')].head(1)

,Titre_ID,Titre,Annee,Duree,Genre,Note,Nb_votes,Langue_Titre,Realisateur,Acteurs,Image,Budget,Resume,Popularite,Affiche,Pays,Recette
190038,tt0211915,Le Fabuleux Destin d'Amélie Poulain,2001,122,"Comedy,Romance",8.3,836238.0,\N,Jean-Pierre Jeunet,"Audrey Tautou, Mathieu Kassovitz, Rufus, Lorel...",/9Y9K6LeLrMeofOvX7hZW36Aj3OG.jpg,10000000.0,"At a tiny Parisian café, the adorable yet pain...",26.104,/oTKduWL2tpIKEmkAqF4mFEAWAsv.jpg,"['FR', 'DE']",173921954.0


In [ ]:
df_final.loc[df_final["Titre"].str.contains('Joker')]

,Titre_ID,Titre,Annee,Duree,Genre,Note,Nb_votes,Langue_Titre,Realisateur,Acteurs,Image,Budget,Resume,Popularite,Affiche,Pays,Recette
1635993,tt11315808,Joker: Folie à Deux,2024,138,"Drama,Musical,Thriller",5.2,179876.0,\N,Todd Phillips,"Joaquin Phoenix, Lady Gaga, Brendan Gleeson, C...",/bDuCriydxdLecnCqMjBEynxw9mR.jpg,150000000.0,A sequel to the 2019 film Joker.,4.766,/hoUe1MpSU8Eji6cxYbOYY6k2DFa.jpg,"['CA', 'US']",0.000000e+00
5363393,tt21651430,The People's Joker,2022,92,Comedy,6.3,2144.0,\N,Vera Drew,"Vera Drew, Lynn Downey, Kane Distler, Nathan F...",/79M8FwF2UtyeFZRF7fRGwMx4pwb.jpg,0.0,An aspiring clown grappling with her gender id...,2.335,/1P74yiStxwtmwHe5q8hNhp3vlXo.jpg,['US'],0.000000e+00
11618300,tt9208444,Impractical Jokers: The Movie,2020,92,Comedy,5.7,7718.0,\N,Chris Henchy,"Paula Abdul, Joe Gatto, James Murray, Brian Qu...",/ummzpRlgm6iSP4wIsTmEZHHlkrG.jpg,3000000.0,The story of a humiliating high school mishap ...,6.796,/kTHzM6pPIjs4LHX33thyZpnKiOP.jpg,['US'],1.070000e+07
10778748,tt7286456,Joker,2019,122,"Crime,Drama,Thriller",8.3,1678237.0,\N,Todd Phillips,"Joaquin Phoenix, Robert De Niro, Zazie Beetz, ...",/n6bUvigpRFqSwmPp1m2YADdbRBc.jpg,55000000.0,"During the 1980s, a failed stand-up comedian i...",58.411,/udDclJoHjfjb8Ekgsd4FDteOkCU.jpg,"['CA', 'US']",1.074458e+09
